# Experiment 4: weather features as changes, and upstream points

**Question:** in Experiment 3, pressure, humidity and wind (setup D) did not help the model see a cold front
coming. There we gave their **levels**. TiRex-2 uses covariates best when they move together with the
temperature, so here we give the same information in a form that moves more:

| Setup | Model input besides temperature (all past covariates, only values up to the forecast start) |
|---|---|
| **A** | nothing (from Experiment 3) |
| **D** | pressure, humidity, wind as levels (from Experiment 3) |
| **G** | pressure, humidity, wind as **changes over the last 3 h** |
| **U** | **upstream points**: average 3 h temperature change of the points 40-120 km and 120-240 km to the north-west |
| **GU** | G and U together |
| **Ua** | like U, but the upstream change **minus the usual 3 h change at that hour and month** (removes the normal evening cooling, so mostly the unusual part is left) |

**Upstream:** our 16 fronts all move towards the south-east (121-191°), as cold fronts in Austria usually do,
so they arrive from the north-west. "Upstream" = points in the sector from west-north-west to north
(bearing 290° to 20°) of the forecast point. Western and northern border points have few or none (the
grid ends at the border); their cases get 0 for the missing ring.

**Cases and measures:** the same 450 cases as Experiment 3 (15 fronts × 30 points, forecast starts 6 h before
the temperature drop, 10 weeks of history), same measures: **drop captured (0-100 %)** and **error (°C)**.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp4")
FIG.mkdir(parents=True, exist_ok=True)
EXP3 = Path("/workspace/data/notebooks/results/exp3_forecasts_30perfront.npz")
C, H, BEFORE = 10 * 168, 24, 6
RINGS = [(40, 120), (120, 240)]  # km
SECTOR = (290, 20)               # bearing from the point, degrees (W-NW to N)
QUICK = bool(os.environ.get("QUICK"))
N_QUICK = 30
if QUICK:  # quick code test
    FIG = Path("/tmp")  # test runs must not overwrite the real figures
RESULTS = Path("/workspace/data/notebooks/results/exp4_forecasts_30perfront.npz")
RERUN = False
SETUPS = {"A": "A: temperature only", "D": "D: pressure, humidity, wind (levels)",
          "G": "G: pressure, humidity, wind (3 h changes)", "U": "U: upstream temperature changes",
          "GU": "GU: changes + upstream", "Ua": "Ua: upstream, unusual part only"}
COLOR = {"A": "#7f7f7f", "D": "#2ca02c", "G": "#17becf", "U": "#8c564b", "GU": "#e377c2", "Ua": "#bcbd22"}

In [2]:
VARS = ["T2M", "P0", "RH2M", "UU", "VV"]
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", *VARS])
wide = {v: df.pivot(index="time", columns="point_id", values=v).interpolate(limit=3, limit_area="inside") for v in VARS}
times, pts = wide["T2M"].index, wide["T2M"].columns
col = {p: i for i, p in enumerate(pts)}
X = {v: w.to_numpy(np.float32) for v, w in wide.items()}
X["P0"] = X["P0"] / 100  # Pa -> hPa
T = X["T2M"]


def change3(a):
    """Change over the LAST 3 h at each hour (uses only the past)."""
    d = np.full_like(a, np.nan)
    d[3:] = a[3:] - a[:-3]
    return d


dX = {v: change3(X[v]) for v in VARS}

z = np.load(EXP3)
cases = pd.DataFrame({"point_id": z["point_id"], "front": pd.to_datetime(z["front"]), "start": pd.to_datetime(z["start"])})
cases["o"] = [times.get_loc(s) for s in cases.start]
truth = z["truth"]
fc = {s: z[s] for s in ["A", "D"]}
if QUICK:
    keep = np.random.default_rng(0).choice(len(cases), N_QUICK, replace=False)
    cases, truth = cases.iloc[keep].reset_index(drop=True), truth[keep]
    fc = {s: v[keep] for s, v in fc.items()}
print(f"{len(cases)} cases from Experiment 3 ({cases.front.nunique()} fronts)")

450 cases from Experiment 3 (15 fronts)


## 1. Upstream points

For every point: the grid points in the north-west sector at 40-120 km (near ring) and 120-240 km (far ring).
The map shows them for one example point.

In [3]:
P = pd.read_csv(DATA / "points_with_labels.csv").set_index("point_id").loc[pts]
xy = P[["x", "y"]].to_numpy() / 1000  # km
dx, dy = xy[None, :, 0] - xy[:, None, 0], xy[None, :, 1] - xy[:, None, 1]  # from row point to column point
dist = np.hypot(dx, dy)
bearing = np.degrees(np.arctan2(dx, dy)) % 360  # 0 = north, 90 = east
in_sector = (bearing >= SECTOR[0]) | (bearing <= SECTOR[1])
ring_idx = [[np.where(in_sector[i] & (dist[i] >= lo) & (dist[i] < hi))[0] for i in range(len(pts))] for lo, hi in RINGS]

# upstream signal: average 3 h temperature change of the ring's points (0 if the ring is empty)
up = np.zeros((len(RINGS), *T.shape), np.float32)
for r, idx in enumerate(ring_idx):
    for i, js in enumerate(idx):
        if len(js):
            up[r, :, i] = dX["T2M"][:, js].mean(axis=1)
up = np.nan_to_num(up)
# unusual part: minus the mean change of that point at that month and hour (climatology over the year)
dTa = dX["T2M"] - pd.DataFrame(dX["T2M"]).groupby([times.month, times.hour]).transform("mean").to_numpy()
up_a = np.zeros_like(up)
for r, idx in enumerate(ring_idx):
    for i, js in enumerate(idx):
        if len(js):
            up_a[r, :, i] = dTa[:, js].mean(axis=1)
up_a = np.nan_to_num(up_a)

n_up = np.array([[len(ring_idx[r][col[p]]) for r in range(len(RINGS))] for p in cases.point_id])
cases["n_near"], cases["n_far"] = n_up[:, 0], n_up[:, 1]
print(f"cases with points in the near ring: {(cases.n_near > 0).mean():.0%}, in the far ring: {(cases.n_far > 0).mean():.0%}, "
      f"in both: {((cases.n_near > 0) & (cases.n_far > 0)).mean():.0%}")

ex = col[cases.point_id[np.argmax(cases.n_near.to_numpy() + cases.n_far.to_numpy())]]
role = np.full(len(pts), "other point", dtype=object)
role[ring_idx[1][ex]], role[ring_idx[0][ex]], role[ex] = "far ring (120-240 km)", "near ring (40-120 km)", "forecast point"
fig = go.Figure()
for name, colr, size in [("other point", "#5a5a5a", 6), ("far ring (120-240 km)", "#c49c94", 8),
                         ("near ring (40-120 km)", "#8c564b", 9), ("forecast point", "white", 13)]:
    m = role == name
    fig.add_trace(go.Scattergeo(lon=P.lon[m], lat=P.lat[m], mode="markers", name=name,
                                marker={"color": colr, "size": size, "line": {"width": 0}}, hoverinfo="skip"))
fig.update_geos(fitbounds="locations", showcountries=True, countrycolor="#666", showland=True, landcolor="#1e1e1e", bgcolor="rgba(0,0,0,0)",
                showlakes=False, resolution=50)
fig.update_layout(height=430, template="plotly_dark", margin={"l": 0, "r": 0, "t": 60, "b": 0},
                  title="Upstream points of one forecast point<br><sup>fronts arrive from the north-west; "
                        "the model gets the average 3 h temperature change of each ring</sup>")
fig.write_html(FIG / "exp4_upstream_map.html", include_plotlyjs="cdn")
fig.show()

cases with points in the near ring: 82%, in the far ring: 57%, in both: 57%


## 2. Forecasts

In [4]:
def zscore(x):
    x = np.nan_to_num(x)
    return (x - x.mean(axis=1, keepdims=True)) / np.maximum(x.std(axis=1, keepdims=True), 1e-3)


def series(setup, p, o):
    c = col[p]
    parts = []
    if "G" in setup:
        parts += [dX[v][o - C : o, c] for v in ["P0", "RH2M", "UU", "VV"]]
    if setup == "Ua":
        parts += [up_a[r, o - C : o, c] for r in range(len(RINGS))]
    elif "U" in setup:
        parts += [up[r, o - C : o, c] for r in range(len(RINGS))]
    past = zscore(np.stack(parts))
    t = lambda a: torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))
    return TimeseriesType(t(T[o - C : o, c][None]), t(past), None)


model = load_model("NX-AI/TiRex-2", device="cpu")
if RESULTS.exists() and not RERUN and not QUICK:
    zz = np.load(RESULTS)
    assert list(zz["point_id"]) == list(cases.point_id)
    fc.update({s: zz[s] for s in SETUPS if s in zz.files})
    print(f"loaded saved forecasts from {RESULTS.name}")
for s in [s for s in SETUPS if s not in fc]:
    t0 = time.time()
    out = model.forecast([series(s, p, o) for p, o in zip(cases.point_id, cases.o)],
                         prediction_length=H, output_type="numpy", batch_size=32)
    fc[s] = np.stack([q[0] for q in out])
    print(f"{SETUPS[s]:45s} {time.time() - t0:5.0f} s ({1000 * (time.time() - t0) / len(cases):.0f} ms per forecast)")
if not QUICK:
    np.savez_compressed(RESULTS, point_id=cases.point_id.to_numpy(), front=cases.front.astype(str).to_numpy().astype("U"),
                        start=cases.start.astype(str).to_numpy().astype("U"), truth=truth,
                        n_near=cases.n_near.to_numpy(), n_far=cases.n_far.to_numpy(),
                        **{s: fc[s] for s in SETUPS if s not in ("A", "D")})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp4_forecasts_30perfront.npz


## 3. Results

In [5]:
W = slice(BEFORE, BEFORE + 7)  # drop start and the 6 h after it
real_drop = truth[:, BEFORE] - truth[:, W].min(axis=1)
ok = real_drop >= 2
has_up = (cases.n_near > 0).to_numpy()


def captured(s):
    med = fc[s][:, 4]
    pred = med[:, BEFORE] - med[:, W].min(axis=1)
    return 100 * np.clip(pred / np.where(ok, real_drop, np.nan), 0, 1)


res = pd.DataFrame({"setup": [SETUPS[s] for s in SETUPS],
                    "drop captured, mean (%)": [np.nanmean(captured(s)[ok]) for s in SETUPS],
                    "... only cases with upstream points (%)": [np.nanmean(captured(s)[ok & has_up]) for s in SETUPS],
                    "error (°C)": [np.abs(fc[s][:, 4] - truth).mean() for s in SETUPS],
                    "mean change vs A (°C)": [np.abs(fc[s][:, 4] - fc["A"][:, 4]).mean() for s in SETUPS]})
print(f"{ok.sum()} cases with a real drop of at least 2 °C, {(ok & has_up).sum()} of them with upstream points")
res.round(2)

269 cases with a real drop of at least 2 °C, 225 of them with upstream points


,setup,"drop captured, mean (%)",... only cases with upstream points (%),error (°C),mean change vs A (°C)
0,A: temperature only,21.070000,20.400000,3.87,0.00
1,"D: pressure, humidity, wind (levels)",20.959999,20.340000,3.93,0.16
2,"G: pressure, humidity, wind (3 h changes)",21.000000,20.400000,3.89,0.10
3,U: upstream temperature changes,20.930000,20.260000,3.89,0.11
4,GU: changes + upstream,21.030001,20.389999,3.88,0.09
5,"Ua: upstream, unusual part only",20.969999,20.309999,3.93,0.12


In [6]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35], horizontal_spacing=0.1,
                    subplot_titles=["How much of the real drop does the forecast show?", "Error over the 24 h"])
for s, label in SETUPS.items():  # short letters on the axes, full description once in the legend
    v = captured(s)[ok]
    fig.add_trace(go.Box(x=[s] * len(v), y=v, name=label, legendgroup=s, marker={"color": COLOR[s], "size": 3}, line={"color": COLOR[s]},
                         boxpoints="all", jitter=0.4, pointpos=0, hovertemplate="%{y:.0f} %<extra>" + label + "</extra>"), row=1, col=1)
    fig.add_trace(go.Bar(x=[s], y=[np.abs(fc[s][:, 4] - truth).mean()], name=label, legendgroup=s, marker_color=COLOR[s], showlegend=False,
                         hovertemplate="%{y:.2f} °C<extra>" + label + "</extra>"), row=1, col=2)
fig.update_yaxes(title_text="drop captured (%)", range=[-3, 103], row=1, col=1)
fig.update_yaxes(title_text="mean error (°C)", rangemode="tozero", row=1, col=2)
fig.update_xaxes(title_text="setup", row=1, col=1)
fig.update_xaxes(title_text="setup", row=1, col=2)
fig.update_layout(height=560, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"Do changes or upstream points help the model see a cold front?<br><sup>{ok.sum()} front passages, "
                        "forecast starts 6 h before the drop, 10 weeks of history. One dot per case.</sup>")
fig.write_html(FIG / "exp4_drop_captured.html", include_plotlyjs="cdn")
fig.show()

## 4. The average front

All cases aligned on the start of the drop (hour 0); temperature change since the forecast started, averaged over
all cases. White = what happened, colours = mean forecast of each setup.

In [7]:
x = np.arange(H) - BEFORE
last = np.array([T[o - 1, col[p]] for p, o in zip(cases.point_id, cases.o)])[:, None]
fig = go.Figure()
fig.add_vrect(x0=0, x1=6, fillcolor="#1f77b4", opacity=0.07, line_width=0, annotation_text="drop window", annotation_position="top left")
fig.add_trace(go.Scatter(x=x, y=(truth - last).mean(axis=0), name="what happened", line={"color": "white", "width": 3.5}))
for s, label in SETUPS.items():
    fig.add_trace(go.Scatter(x=x, y=(fc[s][:, 4] - last).mean(axis=0), name=label, line={"color": COLOR[s], "width": 2.2},
                             hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>" + label + "</extra>"))
fig.update_layout(height=460, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"The average cold front ({len(cases)} cases) and what each setup forecasts",
                  xaxis={"title": "hours from the start of the drop", "dtick": 3},
                  yaxis={"title": "change since forecast start (°C)"})
fig.write_html(FIG / "exp4_average_front.html", include_plotlyjs="cdn")
fig.show()

## Reusing the results

New forecasts are saved in `data/notebooks/results/exp4_forecasts_30perfront.npz` (A and D are in the
Experiment 3 file, same cases in the same order):

```python
z = np.load("/workspace/data/notebooks/results/exp4_forecasts_30perfront.npz")
z["point_id"], z["front"], z["start"]  # one entry per case
z["truth"]                             # (cases, 24) what happened, °C; the drop starts at hour index 6
z["n_near"], z["n_far"]                # number of upstream points per ring (0 = none)
z["G"], z["U"], z["GU"], z["Ua"]       # (cases, 9 quantiles 10-90 %, 24 h) forecasts per setup
```